In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

In [ ]:
!nvidia-smi

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


sys.path.insert(1, '../src/')

from cellpose import models, core, io, plot
from Basic import *
from image_lib import *

In [ ]:
## Check if has access to GPU

In [ ]:
is_laptop = True

root0 = "../../colaboracoes/deOcesano/samples/"

os.listdir(root0)

In [ ]:
plates = [x for x in os.listdir(root0) if x.startswith('Plate')]
plates

In [ ]:
i=-1
plate = plates[i]
plate

In [ ]:
root_plate = os.path.join(root0, plate)

root_exps = [x for x in os.listdir(root_plate) if os.path.isdir( os.path.join(root_plate, x) )]
root_exps

print(root_exps)

root_plate = os.path.join(root_plate, root_exps[0])

## Choose an image to classifier

In [ ]:
image_chose = os.listdir(root_plate)[0]
image_path = os.path.join(root_plate, image_chose)

print(image_chose)
img = io.imread(image_path)
plt.imshow(img)
img.shape

## Cellpose

In [ ]:
gpu_disponible = torch.cuda.is_available()

if gpu_disponible: print('GPU disponível')
else: print('GPU não disponível')

In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '2' # @param ['None', 0, 1, 2, 3, 4, 5]

selected_channels = []

for c in [first_channel, second_channel, third_channel]:
    if c == 'None': continue
    if int(c) > img.shape[-1]: assert False, 'invalid channel index, must have index greater or equal to the number of channels'
    if c != 'None': selected_channels.append(int(c))

selected_channels

### Processing Cellpose

In [ ]:
%%time

img_selected_channels = np.zeros_like(img)
img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]

flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(
    img_selected_channels,
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize}
)